# 06 · EDA

Steps 1–4 of the README's EDA plan:

1. **Collection**: how much data per game, language and time.
2. **Games over time**: online players, reviews and positive share per week, with patch dates.
3. **Cleaning**: what each rule removes.
4. **Review slices**: 👍/👎 by game, language, genre, text length, playtime, helpfulness.
5. **Review types**: themes from `04` (named in `05`), how positive they are, which languages write about them, how they change around patches.

Everything is loaded without review texts, so it fits in memory with millions of reviews.

In [ ]:
import textwrap

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import EngFormatter, PercentFormatter

from src.viz.charts import use_cjk_font
from src.config import CONFIG, TOPIC_NOISE, TOPIC_SHORT, game_names
from src.db import read
from src.text.topics import topic_names

pd.set_option("display.max_colwidth", 120)
# fixed color per language (identity follows the entity, not the rank)
LANG_COLORS = {"english": "#2a78d6", "russian": "#eb6834", "schinese": "#1baf7a", "tchinese": "#eda100"}
LANGS = CONFIG.reviews.languages
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e5e5e5", "grid.linewidth": 0.8, "lines.linewidth": 2, "figure.figsize": (11, 4),
})
use_cjk_font()  # unnamed topics show their keywords, some of them Chinese
SERIES_COLOR = "#2a78d6"
PATCH_COLOR = "#3c3c3c"
# charts start at since_date if set, otherwise at each game's first data
SINCE = pd.Timestamp(CONFIG.reviews.since_date or "2000-01-01")

In [ ]:
games = pd.Series(game_names())
# millions of reviews: only the columns used below, text length instead of the texts
reviews = read.reviews(
    columns=["recommendation_id", "appid", "language", "created_at", "updated_at", "voted_up", "votes_up",
             "author_playtime_at_review", "received_for_free", "steam_purchase",
             "written_during_early_access", "primarily_steam_deck"],
    flag_columns=["text_language", "is_edited", "keep"],
    with_topics=True,
    text_length=True,
)
news = read.news()
players = read.player_counts()

reviews["game"] = reviews["appid"].map(games)
pd.Series({"games": len(games), "reviews": len(reviews), "news": len(news),
           "patches": news["is_patch"].sum(), "player count points": len(players)})

## 1. Collection

### 1.1 Reviews per game × language, and coverage against Steam's totals


In [ ]:
ours = reviews.groupby(["game", "appid", "language"]).agg(
    collected=("recommendation_id", "size"),
    first_review=("created_at", "min"),
    last_review=("created_at", "max"),
    share_positive=("voted_up", "mean"),
)
steam_total = (
    read.review_totals()
    .sort_values("fetched_at")
    .groupby(["appid", "language"])["total_reviews"]
    .last()
    .rename("steam_total")
)
coverage = ours.join(steam_total, on=["appid", "language"])
coverage["coverage"] = coverage["collected"] / coverage["steam_total"]
coverage.round({"share_positive": 3, "coverage": 3})

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(coverage)

In [ ]:
per_game = reviews.groupby(["game", "language"]).size().unstack(fill_value=0)
per_game = per_game[[c for c in CONFIG.reviews.languages if c in per_game]]  # drop odd labels
per_game = per_game.loc[per_game.sum(axis=1).sort_values().index]

ax = per_game.plot.barh(
    stacked=True, color=[LANG_COLORS.get(c, "#888888") for c in per_game.columns],
    edgecolor="white", linewidth=2, width=0.7, figsize=(10, 0.5 * len(per_game) + 1),
)
ax.set(title="Collected reviews per game", xlabel="reviews", ylabel="")
ax.set_axisbelow(True)
ax.grid(axis="y", visible=False)
ax.legend(title="", frameon=False, loc="lower right");

### 1.2 Volume over time

In [ ]:
month = reviews["created_at"].dt.tz_convert(None).dt.to_period("M").dt.to_timestamp()
monthly = reviews.groupby([month, "language"]).size().unstack(fill_value=0)

ax = monthly.plot(color=[LANG_COLORS.get(c, "#888888") for c in monthly.columns])
ax.set(title="Reviews per month by language (all games)", xlabel="", ylabel="reviews")
ax.legend(title="", frameon=False);

### 1.3 Text length and author fields

In [ ]:
reviews.groupby("language")["text_len"].describe(percentiles=[0.1, 0.5, 0.9]).round(0)

In [ ]:
reviews.groupby("language").agg(
    median_hours_at_review=("author_playtime_at_review", lambda s: s.median() / 60),
    share_free=("received_for_free", "mean"),
    share_steam_purchase=("steam_purchase", "mean"),
).round(3)

### 1.4 Steam language vs language of the text


In [ ]:
pd.crosstab(reviews["language"], reviews["text_language"], normalize="index").round(3)

### 1.5 News and online player data

In [ ]:
pd.DataFrame({
    "news": news.groupby("appid").size(),
    "patches": news.groupby("appid")["is_patch"].sum(),
    "first_news": news.groupby("appid")["published_at"].min().dt.date,
    "player_points": players.groupby("appid").size(),
    "players_from": players.groupby("appid")["ts"].min().dt.date,
    "players_to": players.groupby("appid")["ts"].max().dt.date,
}).rename(index=games)

## 2. Games over time, with patches

In [ ]:
def to_naive(ts):
    return ts.dt.tz_convert(None)

patches = news[news["is_patch"].astype(bool)].assign(day=lambda d: to_naive(d["published_at"]).dt.floor("D"))
patch_days = patches.groupby("appid")["day"].unique()


def per_game_panels(data, title: str, ylabel: str, ylim=None, percent=False):
    """data (Series, or DataFrame for several lines) has a MultiIndex (appid, date):
    one panel per game with its patch dates."""
    appids = [a for a in games.index if a in data.index.get_level_values(0)]
    if not appids:
        print(f"{title}: no data yet")
        return
    rows = (len(appids) + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(12, 2.4 * rows), squeeze=False)
    for ax, appid in zip(axes.flat, appids):
        d = data.loc[appid]
        start, end = d.index.min(), d.index.max()
        for day in patch_days.get(appid, []):
            if start <= day <= end:
                ax.axvline(day, color=PATCH_COLOR, linewidth=0.8, alpha=0.5, zorder=0)
        if isinstance(d, pd.Series):
            ax.plot(d.index, d.values, color=SERIES_COLOR)  # NaN = no data: the line breaks
        else:
            for column in d.columns[d.notna().any()]:
                ax.plot(d.index, d[column], label=textwrap.shorten(column, 40))
            ax.legend(fontsize=7, frameon=False, loc="upper left")
        ax.set_title(games[appid], loc="left", fontsize=10)
        ax.set_ylabel(ylabel)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0) if percent else EngFormatter(sep=""))
        locator = mdates.AutoDateLocator()
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        if ylim:
            ax.set_ylim(*ylim)
    for ax in axes.flat[len(appids):]:
        ax.set_visible(False)
    patch_handle = Line2D([], [], color=PATCH_COLOR, linewidth=0.8, label="patch / update")
    fig.legend(handles=[patch_handle], loc="upper right", frameon=False)
    fig.suptitle(title, x=0.01, ha="left")
    fig.tight_layout()


def weekly_with_gaps(s):
    """(appid, week) series -> NaN for missing weeks, so lines break instead of bridging gaps."""
    return s.groupby(level=0, group_keys=True).apply(lambda g: g.droplevel(0).asfreq("W-MON"))

### 2.1 Online players (SteamCharts)


In [ ]:
recent = players[(to_naive(players["ts"]) >= SINCE) & (players["players"] > 0)]
daily_players = (
    recent.assign(day=to_naive(recent["ts"]).dt.floor("D"))
    .groupby(["appid", "day"])["players"]
    .mean()
)
per_game_panels(daily_players, "Online players (daily average)", "players")

### 2.2 Reviews per week

In [ ]:
week = to_naive(reviews["created_at"]).dt.to_period("W").dt.start_time
recent_reviews = reviews[week >= SINCE]
recent_week = week[week >= SINCE]
weekly = recent_reviews.groupby(["appid", recent_week]).agg(n=("voted_up", "size"), positive=("voted_up", "mean"))
per_game_panels(weekly_with_gaps(weekly["n"]), "Reviews per week (all languages)", "reviews")

### 2.3 Share of positive reviews per week

Only weeks with at least 20 reviews, so single reviews don't swing the line.

In [ ]:
per_game_panels(
    weekly_with_gaps(weekly["positive"].where(weekly["n"] >= 20)),
    "Share of positive reviews per week (all languages)", "positive", ylim=(0, 1), percent=True,
)

## 3. Cleaning

### 3.1 How much each rule removes (latest run)

In [ ]:
report = read.cleaning_report()
latest = report[report["run_id"] == report["run_id"].max()]
counts = latest.groupby(["rule", "language"])[["n_flagged", "n_total"]].sum()
(counts["n_flagged"] / counts["n_total"]).unstack("language").style.format("{:.1%}")

### 3.2 Examples per rule

Check by eye that each rule catches what it should.

In [ ]:
def examples(rule: str, n: int = 8) -> pd.DataFrame:
    return read.flagged_examples(rule, n).assign(
        review=lambda d: d["review"].str.replace(r"\s+", " ", regex=True).str.slice(0, 120)
    )

for rule in ["is_empty", "is_junk", "is_duplicate_text", "is_lang_mismatch"]:
    print(f"\n=== {rule}")
    display(examples(rule))

### 3.3 Edited reviews: which date to use

In [ ]:
days_to_edit = (reviews["updated_at"] - reviews["created_at"]).dt.total_seconds() / 86400
pd.DataFrame({
    "share_edited": reviews.groupby("language")["is_edited"].mean(),
    "median_days_to_edit": days_to_edit[reviews["is_edited"].eq(True)].groupby(reviews["language"]).median(),
}).round(3)

### 3.4 Result

In [ ]:
reviews.groupby("language")["keep"].agg(total="size", kept="sum", share_kept="mean").round(3)

## 4. Review slices (👍/👎)

All reviews, not only kept ones: a vote counts even when its text is junk or empty.

### 4.1 Share of positive reviews by game and language

In [ ]:
positive = reviews.pivot_table(index="game", columns="language", values="voted_up", aggfunc="mean")
positive = positive[[c for c in LANGS if c in positive]]
positive.insert(0, "all", reviews.groupby("game")["voted_up"].mean())
positive.sort_values("all").style.format("{:.0%}", na_rep="").background_gradient(cmap="RdYlGn", vmin=0.2, vmax=1)

### 4.2 By Steam genre and category

Share of positive reviews over all reviews of the games that have the genre / category (a game has several). With few games per genre, read it as a hint, not a result.

In [ ]:
per_game = reviews.groupby("appid")["voted_up"].agg(reviews="size", positive="sum")
tags = read.games().set_index("appid")[["genres", "categories"]].join(per_game, how="inner")


def positive_by_tag(column):
    t = tags.explode(column).groupby(column).agg(games=("reviews", "size"), reviews=("reviews", "sum"),
                                                  positive=("positive", "sum"))
    return t.assign(share=t["positive"] / t["reviews"])


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, column, top in [(axes[0], "genres", None), (axes[1], "categories", 15)]:
    t = positive_by_tag(column)
    t = (t.nlargest(top, "games") if top else t).sort_values("share")
    ax.barh(t.index, t["share"], color=SERIES_COLOR)
    for y, (share, n) in enumerate(zip(t["share"], t["games"])):
        ax.text(share, y, f" {share:.0%} ({n} games)", va="center", fontsize=8)
    ax.set_xlim(0, 1.2)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f"Positive share by {column[:-1]}" + (f" (top {top} by games)" if top else ""), loc="left")
    ax.grid(axis="y", visible=False)
fig.tight_layout()

### 4.3 By text length, playtime and helpfulness

One line per Steam language. Bins with few reviews (the long tails) are noisy.

In [ ]:
def positive_by_bins(ax, values, edges, labels, title):
    binned = pd.cut(values, bins=edges, labels=labels, right=False)
    for lang in LANGS:
        mine = reviews["language"].eq(lang)
        share = reviews.loc[mine, "voted_up"].groupby(binned[mine], observed=False).mean()
        ax.plot(range(len(labels)), share.to_numpy(), marker="o", color=LANG_COLORS.get(lang), label=lang)
    ax.set_xticks(range(len(labels)), labels, rotation=45, fontsize=8)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(title, loc="left")


fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
positive_by_bins(axes[0], reviews["text_len"], [0, 20, 50, 100, 200, 500, 1000, 2000, np.inf],
                 ["<20", "20–50", "50–100", "100–200", "200–500", "500–1k", "1k–2k", "2k+"], "Text length (characters)")
positive_by_bins(axes[1], reviews["author_playtime_at_review"] / 60, [0, 1, 2, 5, 10, 20, 50, 100, 200, 500, np.inf],
                 ["<1", "1–2", "2–5", "5–10", "10–20", "20–50", "50–100", "100–200", "200–500", "500+"],
                 "Hours played when reviewing")
positive_by_bins(axes[2], reviews["votes_up"], [0, 1, 2, 6, 21, 101, np.inf],
                 ["0", "1", "2–5", "6–20", "21–100", "100+"], "Helpful votes from other players")
axes[0].set_ylabel("positive")
axes[0].legend(frameon=False)
fig.tight_layout()

### 4.4 Early access, Steam Deck, free copies

In [ ]:
GROUPS = {
    "written_during_early_access": "written in early access",
    "primarily_steam_deck": "played mostly on Steam Deck",
    "received_for_free": "got the game for free",
    "steam_purchase": "bought on Steam",
}
rows = []
for column, name in GROUPS.items():
    yes = reviews[column].eq(True)
    rows.append({"group": name, "reviews": yes.sum(), "share of reviews": yes.mean(),
                 "positive if yes": reviews.loc[yes, "voted_up"].mean(),
                 "positive if no": reviews.loc[~yes, "voted_up"].mean()})
pd.DataFrame(rows).set_index("group").style.format(
    {"reviews": "{:,}", "share of reviews": "{:.1%}", "positive if yes": "{:.0%}", "positive if no": "{:.0%}"}
)

## 5. Review types

Topics from `04_review_types`, named in `05_topics` (until then a topic shows as its first keywords). Topics with the same label, one theme in several languages, count together as one **theme**. Only kept reviews have a topic; shares below are among reviews that have a theme (not too short, not noise).

In [ ]:
topics = read.topics()
has_topics = bool((topics["topic_id"] >= 0).any())
if not has_topics:
    print("no topics yet: run 04_review_types first")
else:
    themes = reviews.loc[reviews["topic_id"].notna(), ["appid", "game", "created_at", "voted_up", "text_language", "topic_id"]]
    print(f"{len(themes):,} reviews with a topic: {themes['topic_id'].eq(TOPIC_SHORT).mean():.0%} too short, "
          f"{themes['topic_id'].eq(TOPIC_NOISE).mean():.0%} no clear theme")
    themes = themes[themes["topic_id"] >= 0].assign(theme=lambda d: d["topic_id"].map(topic_names(topics, with_id=False)))
    stats = themes.groupby("theme")["voted_up"].agg(reviews="size", positive="mean")
    stats["share"] = stats["reviews"] / len(themes)
    print(f"{len(themes):,} reviews in {len(stats)} themes")

### 5.1 Most common themes (bar color: share of positive reviews)

In [ ]:
if has_topics:
    top = stats.nlargest(25, "reviews").iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, 9))
    ax.barh(top.index.map(lambda s: textwrap.shorten(s, 55)), top["share"], color=plt.cm.RdYlGn(top["positive"]))
    for y, (share, pos) in enumerate(zip(top["share"], top["positive"])):
        ax.text(share, y, f" {share:.1%} · {pos:.0%} positive", va="center", fontsize=8)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlim(0, top["share"].max() * 1.35)
    ax.grid(axis="y", visible=False)
    ax.set_title("Share of reviews with a theme", loc="left")

### 5.2 Most negative and most positive themes

Themes with at least 0.5% of reviews.

In [ ]:
if has_topics:
    common = stats[stats["share"] >= 0.005].sort_values("positive")
    display(pd.concat([common.head(12), common.tail(8)]).style.format(
        {"reviews": "{:,}", "positive": "{:.0%}", "share": "{:.1%}"}
    ).background_gradient(subset="positive", cmap="RdYlGn", vmin=0, vmax=1))

### 5.3 Themes one language writes about much more

Share of each theme within each text language; listed when one language's share is at least 3× the average of the other two (and at least 0.5%). Often a game reviewed mostly in that language, local problems (servers, translation, prices) or slang.

In [ ]:
if has_topics:
    by_lang = pd.crosstab(themes["theme"], themes["text_language"], normalize="columns")
    by_lang = by_lang[[c for c in ["en", "ru", "zh"] if c in by_lang]]
    ratio = pd.DataFrame({lang: by_lang[lang] / by_lang.drop(columns=lang).mean(axis=1) for lang in by_lang})
    specific = by_lang.assign(mostly=ratio.idxmax(axis=1), ratio=ratio.max(axis=1))
    specific = specific[(specific["ratio"] >= 3) & (by_lang.max(axis=1) >= 0.005)].sort_values("ratio", ascending=False)
    display(specific.head(25).style.format({"en": "{:.1%}", "ru": "{:.1%}", "zh": "{:.1%}", "ratio": "{:.1f}×"}))

### 5.4 Themes over time, with patches

Weekly share of each game's 4 biggest themes, for the `TOPIC_GAMES` games with the most reviews (weeks with at least 30 reviews).

In [ ]:
TOPIC_GAMES = 6
if has_topics:
    week = to_naive(themes["created_at"]).dt.to_period("W").dt.start_time
    biggest = themes["appid"].value_counts().index[:TOPIC_GAMES]
    recent = themes["appid"].isin(biggest) & (week >= SINCE)
    counts = themes[recent].groupby(["appid", week[recent], "theme"]).size()
    totals = counts.groupby(level=[0, 1]).transform("sum")
    shares = (counts / totals)[totals >= 30].unstack("theme")
    top_themes = themes[recent].groupby("appid")["theme"].agg(lambda s: s.value_counts().index[:4].tolist())
    for appid, keep in top_themes.items():  # each game: only its own top themes
        shares.loc[appid, shares.columns.difference(keep)] = np.nan
    per_game_panels(weekly_with_gaps(shares), "Weekly share of the biggest themes", "share", percent=True)